# 02 — Setup | Deploy and Start the Jobs (from this Git folder)

**Project:** MLB Performance Intelligence Platform

Run this notebook once from the Git folder (`Workspace → Create → Git folder` with this repository).
It uses the notebook's own workspace session (no tokens) to:

1. Create or update **`mlb_performance_intelligence_setup`** (catalog, schemas, Volumes, metadata tables) and run it.
2. Create or update **`mlb_performance_intelligence_daily`** from `resources/jobs/mlb_performance_daily.job.json`,
   pointing every task at this Git folder, with the daily 06:07 America/Santo_Domingo schedule, e-mail alerts on
   failure and a one-hour duration warning.
3. Optionally start the historical load (`mode = backfill`, seasons 2024–2026).

Re-running is safe: existing jobs are updated in place (`jobs/reset`), never duplicated.

In [ ]:
dbutils.widgets.dropdown("run_setup", "true", ["true", "false"], "Run the setup job now")
dbutils.widgets.dropdown("start_backfill", "true", ["true", "false"], "Start the 2024-2026 backfill")
dbutils.widgets.text("seasons", "2024,2025,2026", "Backfill seasons")

In [ ]:
import json, time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
me = w.current_user.me().user_name
nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
ROOT = nb_path.rsplit("/src/", 1)[0]                     # Git folder path, e.g. /Users/<me>/MLB-Daily-Games-ETL-Pipeline
host = w.config.host.rstrip("/")
print(f"User: {me}\nGit folder: {ROOT}")

def notebook(rel):
    return f"{ROOT}/src/{rel}"

def upsert(spec):
    """Create the job, or reset it in place if a job with the same name exists."""
    existing = [j for j in w.jobs.list(name=spec["name"])]
    if existing:
        job_id = existing[0].job_id
        w.api_client.do("POST", "/api/2.2/jobs/reset", body={"job_id": job_id, "new_settings": spec})
        action = "updated"
    else:
        job_id = w.api_client.do("POST", "/api/2.2/jobs/create", body=spec)["job_id"]
        action = "created"
    print(f"{spec['name']}: {action} → {host}/jobs/{job_id}")
    return job_id

## Job specs

In [ ]:
setup_spec = {
    "name": "mlb_performance_intelligence_setup",
    "description": "One-time setup: catalog mlb_intelligence, schemas bronze/silver/gold/metadata/quality, Volumes and metadata tables.",
    "tags": {"project": "mlb-performance-intelligence", "environment": "development"},
    "max_concurrent_runs": 1,
    "tasks": [
        {"task_key": "create_catalog_schemas", "notebook_task": {"notebook_path": notebook("setup/00_Create_Catalog_Schemas"), "source": "WORKSPACE"}},
        {"task_key": "create_metadata_tables", "depends_on": [{"task_key": "create_catalog_schemas"}],
         "notebook_task": {"notebook_path": notebook("setup/01_Create_Metadata_Tables"), "source": "WORKSPACE"}},
    ],
}

with open(f"/Workspace{ROOT}/resources/jobs/mlb_performance_daily.job.json") as f:
    daily_spec = json.load(f)
for t in daily_spec["tasks"]:
    t["notebook_task"]["notebook_path"] = notebook(t["notebook_task"]["notebook_path"].split("/src/", 1)[1])
daily_spec["email_notifications"] = {"on_failure": [me], "on_duration_warning_threshold_exceeded": [me],
                                     "no_alert_for_skipped_runs": True}
daily_spec["health"] = {"rules": [{"metric": "RUN_DURATION_SECONDS", "op": "GREATER_THAN", "value": 3600}]}

setup_id = upsert(setup_spec)
daily_id = upsert(daily_spec)

## Run setup, then start the historical load

In [ ]:
def wait(run_id, label, poll=20):
    while True:
        r = w.jobs.get_run(run_id)
        state = r.state.life_cycle_state.value
        if state in ("TERMINATED", "SKIPPED", "INTERNAL_ERROR"):
            result = r.state.result_state.value if r.state.result_state else state
            print(f"{label}: {result} ({host}/jobs/runs/{run_id})")
            return result
        time.sleep(poll)

if dbutils.widgets.get("run_setup") == "true":
    run = w.jobs.run_now(job_id=setup_id)
    if wait(run.run_id, "Setup") != "SUCCESS":
        raise RuntimeError("Setup job failed; open the run link above.")

if dbutils.widgets.get("start_backfill") == "true":
    run = w.jobs.run_now(job_id=daily_id, job_parameters={"mode": "backfill", "seasons": dbutils.widgets.get("seasons")})
    print(f"Backfill started: {host}/jobs/{daily_id}/runs/{run.run_id}  (about 10–20 minutes)")

displayHTML(f"""<p><b>Setup job</b>: <a href="{host}/jobs/{setup_id}" target="_blank">{host}/jobs/{setup_id}</a><br>
<b>Daily job</b>: <a href="{host}/jobs/{daily_id}" target="_blank">{host}/jobs/{daily_id}</a> — schedule 06:07 America/Santo_Domingo</p>""")